# Edit the NeurIPS paper figures at publication size

This notebook reflows the two figures used by `splicebert_neurips_2026_revised.tex` at their **actual 3.25-inch column width**. The existing 11-inch canvases become much too small when LaTeX scales them into one column; increasing DPI alone cannot fix that.

## What is editable

- `STYLE`: type sizes, line weights, markers, and semantic colors.
- `CORE_LAYOUT`: every circuit node, label, position, box size, and edge.
- `PHASE_LAYOUT`: panel titles, limits, legend position, and figure geometry.

The numerical inputs are deliberately **not** editable here. Each load cell verifies the immutable source hashes and frozen analysis gates before drawing. Exports go to `figures/splicebert/workshop_edits/` and never overwrite manuscript assets during **Run All**.

From the repository root, launch with:

```bash
uv run --with jupyterlab --with ipykernel jupyter lab notebooks/edit_neurips_paper_figures.ipynb
```

In [ ]:
from copy import deepcopy
from pathlib import Path
import sys
import matplotlib.pyplot as plt


def find_repo(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'figures/splicebert').exists():
            return candidate
    raise RuntimeError('Launch Jupyter from inside the gfm-circuit-discovery repository.')


REPO = find_repo(Path.cwd())
FIGURE_CODE = REPO / 'figures/splicebert'
if str(FIGURE_CODE) not in sys.path:
    sys.path.insert(0, str(FIGURE_CODE))

from editable_paper_figures import (
    DEFAULT_EXPORT_DIR,
    audit_figure,
    default_core_layout,
    default_phase_layout,
    default_style,
    export_figure,
    load_core_data,
    load_phase_data,
    palette_preview,
    promote_to_manuscript,
    render_core_figure,
    render_phase_figure,
)

print(f'Repository: {REPO}')
print(f'Edits export to: {DEFAULT_EXPORT_DIR.relative_to(REPO)}')

## 1. Shared visual system

Edit this cell first. Font sizes are already specified at final physical size, so 7.2 pt here means 7.2 pt in the inserted one-column PDF. The semantic thread remains: **purple = donor–acceptor payload/coupling**, **blue = period-3 coordinate**, **green = intron control**, and **slate = controls or unresolved quantities**.

In [ ]:
STYLE = default_style()

# Typography — these sizes survive unchanged in a 3.25-inch LaTeX column.
STYLE['base_font_pt'] = 7.2
STYLE['title_font_pt'] = 8.0
STYLE['annotation_font_pt'] = 6.4
STYLE['node_title_pt'] = 7.1
STYLE['node_subtitle_pt'] = 6.2

# Drawing weights. PDF and SVG stay vector; PNG exports at 600 dpi.
STYLE['axis_linewidth'] = 0.65
STYLE['data_linewidth'] = 1.35
STYLE['marker_size'] = 3.8
STYLE['grid_linewidth'] = 0.45

# Colors may be edited here. Keep semantic roles stable across both figures.
# STYLE['semantic']['coordinate'] = '#3569A8'
# STYLE['semantic']['payload'] = '#6F4E7C'

palette_preview(STYLE);

## 2. Figure: structured causal core

The original left-to-right diagram contains nine small boxes across one column. The default below reflows the same evidence into two inputs, a vertical convergence/write path, and a separate faithfulness panel.

Each node has:

- `xy`: normalized center position inside panel A;
- `wh`: normalized width and height;
- `title` and `subtitle`: displayed text;
- `role`: semantic color key from `STYLE['semantic']`.

Move boxes or rewrite labels here; do not insert numerical claims into the schematic unless they are loaded from an artifact.

In [ ]:
CORE_DATA = load_core_data()  # verifies all seven recorded source hashes and gates
CORE_LAYOUT = default_core_layout()

# Final physical size in the NeurIPS paper. Increase height, not width, for more room.
CORE_LAYOUT['figsize'] = (3.25, 4.65)

# Examples — uncomment and edit by node id.
# node = next(n for n in CORE_LAYOUT['nodes'] if n['id'] == 'l3h1')
# node['title'] = 'L3H1 writer'
# node['xy'] = (0.50, 0.45)

CORE_LAYOUT['faithfulness_xlim'] = (0, 104)
CORE_LAYOUT['faithfulness_ylim'] = (-0.03, 1.06)

In [ ]:
core_fig = render_core_figure(CORE_DATA, style=STYLE, layout=CORE_LAYOUT)
CORE_QA = audit_figure(core_fig, name='structured causal core')
plt.show()

In [ ]:
CORE_OUTPUTS = export_figure(
    core_fig,
    name='fig_full_circuit_edit',
    sources=CORE_DATA['sources'],
    claim_boundary=(
        'Structured top-ranked causal core inside a broad interaction-dependent computation; '
        'not a complete sparse generator.'
    ),
    style=STYLE,
    layout=CORE_LAYOUT,
)

## 3. Figure: period-3 phase surgery

This renderer keeps the complete four-part evidential argument while shortening labels for a one-column figure. The ribbons in A remain descriptive; B is the preregistered primary contrast; C shows all 19 structured controls; D remains explicitly inconclusive.

In [ ]:
PHASE_DATA = load_phase_data()  # verifies source hashes, package hash, status, and gates
PHASE_LAYOUT = default_phase_layout()

# Edit overall geometry and concise panel titles here.
PHASE_LAYOUT['figsize'] = (3.25, 4.25)
PHASE_LAYOUT['titles']['A'] = 'Register'
PHASE_LAYOUT['titles']['B'] = 'Primary effect'
PHASE_LAYOUT['titles']['C'] = 'Controls'
PHASE_LAYOUT['titles']['D'] = 'Coding boundary'
PHASE_LAYOUT['line_label_mode'] = 'direct'  # change to 'legend' to use legend_location
PHASE_LAYOUT['legend_location'] = 'lower right'

# Axis limits are appearance controls; do not use them to hide observations.
PHASE_LAYOUT['primary_xlim'] = (-0.065, 0.205)
PHASE_LAYOUT['controls_ylim'] = (-0.018, 0.165)
PHASE_LAYOUT['context_xlim'] = (-0.42, 0.44)

In [ ]:
phase_fig = render_phase_figure(PHASE_DATA, style=STYLE, layout=PHASE_LAYOUT)
PHASE_QA = audit_figure(phase_fig, name='period-3 phase surgery')
plt.show()

In [ ]:
PHASE_OUTPUTS = export_figure(
    phase_fig,
    name='fig5_p3_phase_surgery_confirmation_edit',
    sources=PHASE_DATA['sources'],
    claim_boundary=(
        'Confirms present-day phase-aligned period-3 use in CDS blocks; '
        'coding specificity and reading-frame semantics remain unresolved.'
    ),
    style=STYLE,
    layout=PHASE_LAYOUT,
)

## 4. Review and promotion

Before promotion, inspect both the vector PDF and the 600-dpi PNG at 100% size. Check that: text is at least 6 pt; no labels collide; the semantic color thread is intact; confidence intervals and every control point remain visible; and the captions still describe exactly what the artwork shows.

The cell below is safe during **Run All**. Promotion requires changing a Boolean *and* typing the exact confirmation phrase. Existing manuscript assets are copied to a timestamped backup under `output/figure_backups/` before replacement.

In [ ]:
assert not CORE_QA['text_sizes_below_threshold'], CORE_QA
assert not PHASE_QA['text_sizes_below_threshold'], PHASE_QA
print('Publication-size text audit: PASS')
print('Review these vector files before promotion:')
print(' ', CORE_OUTPUTS['pdf'])
print(' ', PHASE_OUTPUTS['pdf'])

In [ ]:
PROMOTE_TO_MANUSCRIPT = False
CONFIRMATION = ''

if PROMOTE_TO_MANUSCRIPT:
    promote_to_manuscript(
        core_outputs=CORE_OUTPUTS,
        phase_outputs=PHASE_OUTPUTS,
        confirmation=CONFIRMATION,  # must equal: PROMOTE REVIEWED FIGURES
    )
else:
    print('Safe mode: manuscript figures were not overwritten.')

### After promotion

Recompile the manuscript and inspect the complete page, not only the standalone figure. The revised figures intentionally keep interpretation in the LaTeX captions rather than squeezing caveats into the artwork. If you decide to make either figure span both columns, change the LaTeX environment to `figure*` and set the corresponding notebook width to approximately `6.75` inches before rebalancing the layout.